In [0]:
%pip install --upgrade kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")
print("Path to dataset files:", path)
import pandas as pd
import json
transactions = pd.read_csv(f"{path}/transactions_data.csv")
cards = pd.read_csv(f"{path}/cards_data.csv")
users = pd.read_csv(f"{path}/users_data.csv")

with open(f"{path}/train_fraud_labels.json", "r") as f:
    raw = json.load(f)

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


100%|██████████| 348M/348M [00:02<00:00, 130MB/s]

Extracting files...


Path to dataset files: /home/spark-660223d0-ce59-4ac1-923d-d0/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1


In [0]:
# Flatten the nested structure: {"target": {"7476734": "No", ...}}
labels = pd.DataFrame(
    list(raw["target"].items()),
    columns=["transaction_id", "is_fraud"]
)
labels["transaction_id"] = labels["transaction_id"].astype(int)
transactions_2010 = transactions[transactions['date'].str.startswith('2010')].copy()
transactions_2010['amount'] = transactions_2010['amount'].str[1:].astype(float)
transactions_2010_spark = spark.createDataFrame(transactions_2010)
spark.sql("CREATE SCHEMA IF NOT EXISTS payments.default")
transactions_2010_spark.write.mode('overwrite').option("overwriteSchema",
"true").saveAsTable("payments.default.pay_transactions")
cards_spark = spark.createDataFrame(cards)
users_spark = spark.createDataFrame(users)
spark.sql("CREATE SCHEMA IF NOT EXISTS payments.default")
cards_spark.write.mode('overwrite').option("overwriteSchema",
"true").saveAsTable("payments.default.pay_cards")
users_spark.write.mode('overwrite').option("overwriteSchema",
"true").saveAsTable("payments.default.pay_users")
labels_spark = spark.createDataFrame(labels)
labels_spark.write.mode('overwrite').option("overwriteSchema",
"true").saveAsTable("payments.default.pay_labels")

In [0]:
transactions_2011 = transactions[transactions['date'].str.startswith('2011')].copy()
transactions_2011['amount'] = transactions_2011['amount'].str[1:].astype(float)
transactions_2011_spark = spark.createDataFrame(transactions_2011)
transactions_2011_spark.write.mode('append').saveAsTable("payments.default.pay_transactions")

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS
SELECT *, CASE WHEN merchant_city = 'ONLINE' THEN 'ONLINE'
ELSE 'OFFLINE'
END AS is_online
FROM payments.default.pay_transactions;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8987381554008595>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REPLACE TABLE payments.default.pay_transactions AS\nSELECT *, CASE WHEN merchant_city = 'ONLINE' THEN 'ONLINE'\nELSE 'OFFLINE'\nEND AS is_online\nFROM payments.default.pay_transactions;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS

WITH city_totals AS (
    SELECT
        merchant_city,
        SUM(amount) AS city_total_amount
    FROM payments.default.pay_transactions
    WHERE merchant_city IS NOT NULL
      AND trim(merchant_city) <> ''
      AND amount IS NOT NULL
    GROUP BY merchant_city
),

city_groups AS (
    SELECT
        merchant_city,
        CASE NTILE(3) OVER (ORDER BY city_total_amount)
            WHEN 1 THEN 'Low Level'
            WHEN 2 THEN 'Middle Level'
            WHEN 3 THEN 'High Level'
        END AS city_amount_level
    FROM city_totals
)

SELECT
    t.*,
    COALESCE(c.city_amount_level, 'Città non disponibile') AS city_amount_level
FROM payments.default.pay_transactions AS t
LEFT JOIN city_groups AS c
    ON t.merchant_city = c.merchant_city;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS

WITH customer_aggregation AS (
    SELECT
        client_id,
        COUNT(*) AS customer_n_transactions,
        SUM(amount) AS customer_total_amount
    FROM payments.default.pay_transactions
    WHERE client_id IS NOT NULL
      AND amount IS NOT NULL
    GROUP BY client_id
),

customer_scores AS (
    SELECT
        client_id,
        customer_n_transactions,
        customer_total_amount,

        NTILE(3) OVER (
            ORDER BY customer_n_transactions
        ) AS frequency_tile,

        NTILE(3) OVER (
            ORDER BY customer_total_amount
        ) AS monetary_tile

    FROM customer_aggregation
),

customer_segments AS (
    SELECT
        client_id,

        CASE
            WHEN frequency_tile = 1 THEN 'Bassa frequenza'
            WHEN frequency_tile = 2 THEN 'Media frequenza'
            WHEN frequency_tile = 3 THEN 'Alta frequenza'
        END AS customer_frequency_level,

        CASE
            WHEN monetary_tile = 1 THEN 'Basso valore'
            WHEN monetary_tile = 2 THEN 'Medio valore'
            WHEN monetary_tile = 3 THEN 'Alto valore'
        END AS customer_monetary_level,

        CASE
            WHEN frequency_tile = 3
             AND monetary_tile = 3
                THEN 'Core Champions'

            WHEN frequency_tile = 3
             AND monetary_tile IN (1, 2)
                THEN 'Frequent Value Seekers'

            WHEN frequency_tile IN (1, 2)
             AND monetary_tile = 3
                THEN 'High-Value Occasional'

            WHEN frequency_tile = 1
             AND monetary_tile = 1
                THEN 'Watchlist'

            ELSE 'Stable'
        END AS customer_boston_class

    FROM customer_scores
)

SELECT
    t.*,

    COALESCE(
        s.customer_frequency_level,
        'Non classificato'
    ) AS customer_frequency_level,

    COALESCE(
        s.customer_monetary_level,
        'Non classificato'
    ) AS customer_monetary_level,

    COALESCE(
        s.customer_boston_class,
        'Non classificato'
    ) AS customer_boston_class

FROM payments.default.pay_transactions AS t

LEFT JOIN customer_segments AS s
    ON t.client_id = s.client_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT
  id, address, gender, birth_month, birth_year, current_age, retirement_age,
  latitude, longitude, credit_score, num_credit_cards,
  TRY_CAST(REGEXP_REPLACE(yearly_income,     '[$,]', '') AS DOUBLE) AS yearly_income,
  TRY_CAST(REGEXP_REPLACE(total_debt,        '[$,]', '') AS DOUBLE) AS total_debt,
  TRY_CAST(REGEXP_REPLACE(per_capita_income, '[$,]', '') AS DOUBLE) AS per_capita_income
FROM payments.default.pay_users;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT *,
  ROUND(total_debt / NULLIF(yearly_income,0), 2)            AS debt_to_income,
  ROUND(yearly_income / NULLIF(per_capita_income,0), 2)     AS income_vs_area,
  retirement_age - current_age                               AS years_to_retirement,
  ROUND(total_debt / NULLIF(num_credit_cards,0), 0)         AS debt_per_card,
  CASE WHEN credit_score < 580 THEN 'Poor'
       WHEN credit_score < 670 THEN 'Fair'
       WHEN credit_score < 740 THEN 'Good'
       WHEN credit_score < 800 THEN 'Very Good'
       ELSE 'Excellent' END                                  AS credit_band,
  CASE WHEN birth_year >= 1997 THEN 'Gen Z'
       WHEN birth_year >= 1981 THEN 'Millennial'
       WHEN birth_year >= 1965 THEN 'Gen X'
       ELSE 'Boomer' END                                     AS generation,
  CASE WHEN credit_score < 620 OR total_debt/NULLIF(yearly_income,0) > 1.5 THEN 'Alto'
       WHEN credit_score < 700 OR total_debt/NULLIF(yearly_income,0) > 0.8 THEN 'Medio'
       ELSE 'Basso' END                                      AS risk_segment
FROM payments.default.pay_users;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_users AS
SELECT *,
  CASE WHEN total_debt > yearly_income * 3
         OR (num_credit_cards >= 6 AND yearly_income < per_capita_income) THEN 'Critical'
       WHEN total_debt > yearly_income * 2
         OR yearly_income > per_capita_income * 5 THEN 'Suspicious'
       ELSE 'Normal' END AS fraud_risk_level
FROM payments.default.pay_users;

num_affected_rows,num_inserted_rows
